# Here in this notebook, i will make a customer funnel journey analysis for an e-commerce dataset, as this can help a company in looking in any issues it have in the purcashing process, and see if the purcahsing is succefful in general.

# In diesem Notebook werde ich eine Customer-Funnel-Journey-Analyse für einen E-Commerce-Datensatz erstellen, da dies einem Unternehmen helfen kann, etwaige Probleme im Kaufprozess zu erkennen und festzustellen, ob der Kaufprozess im Allgemeinen erfolgreich ist.

In [21]:
# upload the dataset and needed libraries for visualization
# Laden Sie den Datensatz und die benötigten Bibliotheken für die Visualisierung hoch.

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
df = pd.read_csv("Customer360Insights.csv")
df.head(3)

,SessionStart,CustomerID,FullName,Gender,Age,CreditScore,MonthlyIncome,Country,State,City,Category,Product,Cost,Price,Quantity,CampaignSchema,CartAdditionTime,OrderConfirmation,OrderConfirmationTime,PaymentMethod,SessionEnd,OrderReturn,ReturnReason
0,2019-01-01 02:42:00,1001,Brittany Franklin,Male,57,780,7591,China,Guangdong,Dongguan,electronics,table fan,30,50,4,Instagram-ads,2019-01-01 02:49:00,True,2019-01-01 03:02:00,Cash On Delivery,2019-01-01 02:53:00,False,NaN
1,2019-01-02 20:35:00,1002,Scott Stewart,Female,69,746,3912,China,Shandong,Yantai,fashion,dress,50,80,6,Google-ads,2019-01-02 20:50:00,True,2019-01-02 20:58:00,Debit Card,2019-01-02 20:54:00,False,NaN
2,2019-01-04 03:11:00,1003,Elizabeth Fowler,Female,21,772,7460,UK,England,Birmingham,toys,plush toy,12,20,2,Facebook-ads,2019-01-04 03:30:00,True,2019-01-04 03:40:00,Cash On Delivery,2019-01-04 03:35:00,False,NaN


In [28]:
# creating the funnel stages
# Erstellung der Funnel-Phasen
funnel_stages = ['Session Start', 'Cart Addition',
                 'Order Confirmation', 'Payment Comleted',
                 'Not returend']

# now, calculating the metrics and adding them to stages
# Jetzt werden die Kennzahlen berechnet und den Phasen hinzugefügt
sessionstarts = df['SessionStart'].count()
cartaddition = df['CartAdditionTime'].notna().sum()
orderconfirmation = df['OrderConfirmationTime'].notna().sum()
paymentcomplete = df[df['OrderConfirmationTime'].notna()]['PaymentMethod'].notna().sum()
notreturned = df[(df['OrderConfirmationTime'].notna()) & (df['OrderReturn'] != True)].shape[0]

stage_counts = [sessionstarts, cartaddition, orderconfirmation,
                paymentcomplete, notreturned]

In [29]:
# building the dataset for the funnel stages
# Erstellung des Datensatzes für die Trichterphasen

df_funnel = pd.DataFrame({'Stage': funnel_stages, 'Count': stage_counts})
df_funnel

,Stage,Count
0,Session Start,2000
1,Cart Addition,2000
2,Order Confirmation,1700
3,Payment Comleted,1700
4,Not returend,1464


In [30]:
# calculate the conversion and dropoff rates
# Konversions- und Abbruchraten berechnen

df_funnel['Conversion rate'] = (df_funnel['Count'] / df_funnel['Count'].iloc[0]) * 100
df_funnel['Dropoff Rate'] = (1 - df_funnel['Count'] /  df_funnel['Count'].shift(1)) * 100
df_funnel['Dropoff Rate'].iloc[0] = 0

# and visualizing the results
# und Ergebnisse visualisieren
fig = go.Figure(go.Funnel(
    y = df_funnel['Stage'], x = df_funnel['Count'],
    textinfo= "value+percent initial", opacity=0.8,
    marker={"color": ["skyblue", "salmon", "tan", "teal", "silver"]}))

fig.update_layout( title="Customer conversion funnel", funnelmode="stack",
                  showlegend=False, height=600)
fig.show()

/tmp/ipython-input-3430039936.py:6: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


/tmp/ipython-input-3430039936.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documen

In [31]:
# making a detailed report about the customer funnel journey in the dataset
# Erstellung eines detaillierten Berichts über den Customer Journey im Datensatz

print(" Funnel analysis results: ")
print(" =" * 20)

for i , row in df_funnel.iterrows():
    print(f"{row['Stage']} {row['Count']:,} users ({row['Conversion rate']:.1f}%)")
    if i > 0:
       print(f" > Dropoff: {row['Dropoff Rate']:.1f}%")

print(" =" * 20)
print(f"Overall Conversion rate: {df_funnel['Conversion rate'].iloc[-1]:.1f}%")

 Funnel analysis results: 
 = = = = = = = = = = = = = = = = = = = =
Session Start 2,000 users (100.0%)
Cart Addition 2,000 users (100.0%)
 > Dropoff: 0.0%
Order Confirmation 1,700 users (85.0%)
 > Dropoff: 15.0%
Payment Comleted 1,700 users (85.0%)
 > Dropoff: 0.0%
Not returend 1,464 users (73.2%)
 > Dropoff: 13.9%
 = = = = = = = = = = = = = = = = = = = =
Overall Conversion rate: 73.2%


# discussing results
we see quiet average rate of dropping from the purchase process after confirming order, and it is possible that some products don't show have information early on which in turn make customers stop buying the cart.

there is also around 13% percentage of people that return the cart after buying, and this need an analysis in the campaign schema type, the products that were returned, and fix the issues with them.

= = = = = = = = = = =

# Diskussion der Ergebnisse
Wir beobachten eine relativ geringe Abbruchrate im Kaufprozess nach der Bestellbestätigung, Es ist möglich, dass bei einigen Produkten die Informationen nicht frühzeitig angezeigt werden, was wiederum dazu führt, dass Kunden den Kauf im Warenkorb abbrechen.

Außerdem gibt es einen Prozentsatz von rund 13 % der Kunden, die ihren Warenkorb nach dem Kauf zurücksenden. Dies erfordert eine Analyse des Kampagnenschemas, der zurückgegebenen Produkte und die Behebung der damit verbundenen Probleme.

Dataset reference:

https://www.kaggle.com/datasets/davedarshan/customer360insights